# Repository edition: 11_adjoints

Run from this repository in your existing GPU environment. Source is in `../suites/11_adjoints/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Frozen/live adjoints, input origins, gate Jacobians and learning
**Source included in this repository; no installs or Hugging Face token cells. No old result ZIP needed.**

Recommended default: **experiments 1 and 2**, 100 matched prompts on Llama1B + Llama8B.
Optional experiments 3, 4 and 5 are ready below. Enable before launching, then run
configuration once. The queue runs adjoints on both models first; expensive optional
jobs follow on 1B. Runtime is not benchmarked; use observed progress.

This tests conditional derivatives, not a validated classifier of causal error
stages. Existing origin labels are included, with uncalibrated cases retained.


In [ ]:
from pathlib import Path
import base64,hashlib,io,json,os,subprocess,sys,time,uuid,zipfile,shutil
BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('11_adjoints'))).hexdigest()
BUNDLE = _source_bundle('11_adjoints')

payload=base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest()==BUNDLE_SHA256
CODE=Path.home()/('threeways_adjoints_code_'+BUNDLE_SHA256[:12]);CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        if CODE.resolve() not in (CODE/item.filename).resolve().parents:raise ValueError('Unsafe archive path')
    z.extractall(CODE)
EVIDENCE=CODE/'evidence'
print('Code ready:',CODE)


## 1. Select your existing GPU environment

In [ ]:
candidates=[Path('/home/ubuntu/venvs/threeways/bin/python'),Path(sys.executable),Path.home()/'threeways_causal_env/bin/python']
check="import torch,transformers,numpy,pandas,matplotlib; assert transformers.__version__=='4.48.3', 'Expected tested transformers 4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON=None;errors=[]
for p in dict.fromkeys(candidates):
    if not p.exists():continue
    result=subprocess.run([str(p),'-c',check],capture_output=True,text=True)
    if result.returncode==0:PYTHON=p;print('Using:',p,'|',result.stdout.strip());break
    errors.append(str(p)+': '+result.stderr[-600:])
if PYTHON is None:raise RuntimeError('Point candidates to your previous working environment.\n'+'\n'.join(errors))


## 2. Configure once
Leave optional flags false for the focused first run. To run everything, set all
three true. No new calibration is silently inferred; bundled regimes may be sparse.

- **1+2:** two backwards/prompt; every position/cut, origin geometry and embedding certificates.
- **3:** eight random VJPs/layer, 100 prompts on 1B; first and persistent crossings.
- **4:** single-matrix CE steps, 100 prompts on 1B; gate/down at three depths, three doses.
- **5:** same-matrix HVPs, 8 prompts on 1B; optional and more memory intensive.

Exp.4 is deliberately **one matrix at a time**. Simultaneous updates to multiple
matrices are not affine, even when all gates/patterns/scales are fixed. A zero content
term for gate weights is structural, not evidence of an absent fact.


In [ ]:
from datetime import datetime
if 'RESULTS' in globals():raise RuntimeError(f'Already configured: {RESULTS}. Use monitor or reconnect, not a new run.')
RUN_JACOBIAN=False
RUN_SGD=False
RUN_CURVATURE=False
CONFIG=json.loads((CODE/'adjoint_study/config.json').read_text())
CONFIG['stages']=['adjoints']
if RUN_JACOBIAN:CONFIG['stages'].append('jacobian')
if RUN_SGD:CONFIG['stages'].append('sgd')
if RUN_CURVATURE:CONFIG['stages'].append('curvature')
# Optional changes BEFORE launch:
# CONFIG['models']=['llama1b']
# CONFIG['optional_models']=['llama1b','llama8b']
# CONFIG['adjoints_prompts']=None  # All 2995 prompts; much more work.
# CONFIG['weight_layer_fractions']=[i/15 for i in range(16)]  # All 1B layers.
# CONFIG['weight_families']=['gate','up','down','q','k','v','o']
# CONFIG['regimes_csv']='/absolute/path/to/your/regimes.csv'
RUN_NAME='adjoints_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULTS=Path.home()/'threeways_adjoints_results'/RUN_NAME
RESULTS.mkdir(parents=True,exist_ok=False)
CONFIG_PATH=RESULTS/'configuration.json';CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
(RESULTS/'bundle.json').write_text(json.dumps({'sha256':BUNDLE_SHA256}))
print('Results:',RESULTS,'\nStages:',CONFIG['stages'])
print('Free disk GiB:',round(shutil.disk_usage(RESULTS).free/2**30,1))


## 3. Launch detached queue

In [ ]:
if (RESULTS/'launch.json').exists():raise RuntimeError('Already launched; monitor or use explicit resume.')
if shutil.disk_usage(RESULTS).free<CONFIG['min_free_gb']*2**30:raise RuntimeError('Insufficient free disk.')
command=[str(PYTHON),'-u','-m','adjoint_study.supervisor','--config',str(CONFIG_PATH),'--out',str(RESULTS),'--evidence',str(EVIDENCE)]
with (RESULTS/'supervisor.log').open('a') as log:
    supervisor=subprocess.Popen(command,cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Supervisor PID:',supervisor.pid)
print('You may close the browser. Keep the GPU instance running.')


## 4. Monitor — rerun any time

In [ ]:
def tail(path,n=6000):
    if path.exists():
        with path.open('rb') as f:
            f.seek(max(0,path.stat().st_size-n));print(f.read().decode(errors='replace'))
tail(RESULTS/'execution.log')
for model in CONFIG['models']:
    for stage in CONFIG['stages']:
        print(model,stage,':',sum(1 for _ in (RESULTS/model/stage).glob('*.json')),'completed prompts')
p=RESULTS/'exit.json'
print('QUEUE EXIT:',json.loads(p.read_text()) if p.exists() else 'Not recorded yet')
print('Completion record:',(RESULTS/'completion.json').exists())
if not (RESULTS/'execution.log').exists():tail(RESULTS/'supervisor.log')


## 5. Refresh reports from completed prompts
Safe while worker runs: makes a separate snapshot folder to avoid concurrent report writes.
Reports retain uncalibrated regimes. Inspect coverage before interpreting differences.


In [ ]:
SNAPSHOT=RESULTS/'snapshots'/datetime.now().strftime('%Y%m%d_%H%M%S')
SNAPSHOT.mkdir(parents=True,exist_ok=False)
for model in CONFIG['models']:
    for stage in CONFIG['stages']:
        dest=SNAPSHOT/model/stage;dest.mkdir(parents=True,exist_ok=True)
        for p in (RESULTS/model/stage).glob('*.json'):shutil.copy2(p,dest/p.name)
subprocess.run([str(PYTHON),'-m','adjoint_study.run','--config',str(CONFIG_PATH),'--out',str(SNAPSHOT),'--evidence',str(EVIDENCE),'--report-only'],cwd=CODE,check=True)
print('Audit:',json.loads((SNAPSHOT/'reports/audit.json').read_text()))
print('Snapshot reports:',SNAPSHOT/'reports')


## 6. Inspect a compact summary and plots

In [ ]:
import csv
from IPython.display import display,Image
REPORTS=(SNAPSHOT if 'SNAPSHOT' in globals() else RESULTS)/'reports'
p=REPORTS/'input_origin_summary.csv'
if p.exists():
    rows=list(csv.DictReader(p.open()))
    for r in rows:
        if r['regime']=='ALL' and r['role'] in ['subject','template','first']:
            print({k:r[k] for k in ['model','role','subjects','selection_fraction','live_frozen_cosine','frozen_origin_margin']})
for p in REPORTS.glob('*_adjoints.png'):display(Image(filename=str(p)))


## 7. Export results (complete or partial)
Includes scalar per-prompt records, reports, metadata, labels and code; no model
weights or activation/gradient caches. If running, use the snapshot cell first.
Download the ZIP before deleting the GPU instance.


In [ ]:
from IPython.display import FileLink
status=json.loads((RESULTS/'exit.json').read_text()) if (RESULTS/'exit.json').exists() else None
complete=bool((RESULTS/'completion.json').exists() and status and status['returncode']==0 and not status['stopped'])
EXPORT_SOURCE=RESULTS if complete else (SNAPSHOT if 'SNAPSHOT' in globals() else RESULTS)
if not complete and EXPORT_SOURCE==RESULTS:raise RuntimeError('Run the snapshot cell first while the queue is active or incomplete.')
archive=RESULTS.parent/(RESULTS.name+('_COMPLETE' if complete else '_SNAPSHOT')+'_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for model in CONFIG['models']:
        for stage in CONFIG['stages']:
            for p in (EXPORT_SOURCE/model/stage).glob('*.json'):z.write(p,p.relative_to(EXPORT_SOURCE))
        p=RESULTS/model/'model.json'
        if p.exists():z.write(p,p.relative_to(RESULTS))
    for p in (EXPORT_SOURCE/'reports').glob('*'):
        if p.is_file():z.write(p,p.relative_to(EXPORT_SOURCE))
    for name in ['configuration.json','protocol.json','environment.json','bundle.json','completion.json','exit.json','regimes.csv']:
        p=RESULTS/name
        if p.exists():z.write(p,'run_metadata/'+name)
    z.write(EVIDENCE/'data/cohort.json','data/cohort.json')
    z.write(EVIDENCE/'regime_provenance.json','data/bundled_regime_provenance.json')
    for p in (RESULTS/'source').rglob('*.py'):z.write(p,p.relative_to(RESULTS))
    z.writestr('EXPORT_MANIFEST.json',json.dumps({'source':str(EXPORT_SOURCE),'run':str(RESULTS),'complete':bool(complete),'status':status,'created':time.time()}))
    for name in ['execution.log','supervisor.log']:
        p=RESULTS/name
        if p.exists():
            with p.open('rb') as f:f.seek(max(0,p.stat().st_size-50000));z.writestr('log_tails/'+name,f.read())
DOWNLOAD=Path.cwd()/archive.name
if archive.resolve()!=DOWNLOAD.resolve():shutil.copy2(archive,DOWNLOAD)
print('ZIP:',archive,'| MB:',round(archive.stat().st_size/2**20,2))
display(FileLink(DOWNLOAD.name))


## Optional: reconnect after restarting the notebook kernel
Run the unpack and environment cells, then this cell INSTEAD of configuration.
Select the exact existing folder; do not start another campaign accidentally.


In [ ]:
# Uncomment and set the real folder:
# RESULTS=Path('/home/ubuntu/threeways_adjoints_results/adjoints_...')
# CONFIG_PATH=RESULTS/'configuration.json'
# CONFIG=json.loads(CONFIG_PATH.read_text())
# assert json.loads((RESULTS/'bundle.json').read_text())['sha256']==BUNDLE_SHA256, 'Use the original notebook for this run'


## Optional: graceful stop / explicit resume

In [ ]:
# Request a graceful stop (between work units):
# (RESULTS/'STOP').touch()
# Then use monitor until exit.json confirms the worker stopped.


In [ ]:
# Set True only to resume the SAME run after its worker has exited.
RESUME=False
if RESUME:
    import fcntl
    locks=[]
    try:
        for name in ['supervisor.lock','run.lock']:
            h=(RESULTS/name).open('a+');locks.append(h)
            fcntl.flock(h,fcntl.LOCK_EX|fcntl.LOCK_NB)
        (RESULTS/'STOP').unlink(missing_ok=True)
    finally:
        for h in locks:h.close()
    command=[str(PYTHON),'-u','-m','adjoint_study.supervisor','--config',str(CONFIG_PATH),'--out',str(RESULTS),'--evidence',str(EVIDENCE),'--resume']
    with (RESULTS/'supervisor.log').open('a') as log:
        subprocess.Popen(command,cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    print('Resume requested; monitor. Completed prompts are reused only under identical protocol.')


## Interpretation and limits
- Origin regimes are observational and have incomplete calibration coverage.
- Answer position is the last prompt token; it overlaps the template role.
- Norm fractions are not additive causal percentages; cross terms are retained.
- Report both crossing probability and depth conditional on crossing; eight probes
  are noisy and token-count-normalized and total comparisons answer different questions.
- Small single-matrix CE steps are not full-sequence fine-tuning or specific fact repair.
- A frozen same-matrix zero Hessian does not imply frozen multi-matrix linearity.
- Finite checks marked unresolved do not provide strong positive certification.

See the included README.txt for exact definitions and tests. Implementation tested
on tiny random Llama/Qwen/OLMo models, not benchmarked on pretrained GH200 runs.
